# HDPM_097 Assignment 2

## Iterative development of simulation model

The aim of this experiment is to see if a LLM can be used to replicate the stroke pathway simulation model described by Monks et Al (2016).

## Iteration 1

In this first attempt, the LLM was used to attempt to model the stroke patient arrivals process using `SimPy`. The LLM used was ChatGPT 5.2 

The prompt and output are available at: https://chatgpt.com/share/69a47680-0fc0-800c-a25a-1e6cd2ff602c

## Prompt:

I want you to help me generate python code to simulate the arrival and flow of patients through a hospital stroke pathway. This will be an iterative process. In this first iteration, please only generate the code to simulate arrivals. Later we model the flow of patients through the hospital wards.

 Use python 3.11 and SimPy 4.1.1 
 
 Patients are classified into 4 groups: Stroke, TIA, Complex neurological and other. Over the base period, the following number of patients were seen:
 - Acute stroke, n = 1320
 - TIA, n = 158
 - complex neurological, n = 456
 - other, n = 510
 Please maintain these proportions in the simulation model.
 
 New patients should arrive with the following mean inter-arrival times:
 - Acute stroke, 1.2 days
 - TIA, 9.3 days
 - Complex neurological, 3.6 days
 - other, 3.2 days
 For all groups, the time between arrival of new admissions should be modelled using an exponential probability distribution.
 
 Construct a function to allow the generation of new patients. The function should accept parameters for the length of the simulation in days, and which categories of patient to yield. The default should be to yield patients from all categories. Include a helper function for sampling from an exponential distribution.


In [8]:
"""
Stroke pathway DES (Iteration 1): arrivals only

Python: 3.11
SimPy:  4.1.1

Arrival processes:
- Stroke: mean IAT 1.2 days
- TIA:    mean IAT 9.3 days
- Complex neurological: mean IAT 3.6 days
- Other:  mean IAT 3.2 days

All IATs ~ Exponential(mean).
"""

from __future__ import annotations

from dataclasses import dataclass
from typing import Iterable, Optional, Sequence

import simpy
import numpy as np

# ---- Configuration ----

CATEGORY_ORDER: tuple[str, ...] = ("stroke", "tia", "complex_neuro", "other")

MEAN_IAT_DAYS: dict[str, float] = {
    "stroke": 1.2,
    "tia": 9.3,
    "complex_neuro": 3.6,
    "other": 3.2,
}

BASE_PERIOD_COUNTS: dict[str, int] = {
    "stroke": 1320,
    "tia": 158,
    "complex_neuro": 456,
    "other": 510,
}


# ---- Data model ----

@dataclass(frozen=True)
class Patient:
    id: int
    category: str
    arrival_time: float  # days since t=0


# ---- Sampling helper ----

def sample_exponential(rng: np.random.Generator, mean: float) -> float:
    """
    Sample from an exponential distribution with the given mean.
    (NumPy uses 'scale' = mean.)
    """
    if mean <= 0:
        raise ValueError("mean must be > 0")
    return float(rng.exponential(scale=mean))


# ---- Arrival generator(s) ----

def _arrival_stream(
    env: simpy.Environment,
    *,
    category: str,
    mean_iat_days: float,
    rng: np.random.Generator,
    patients_out: list[Patient],
    id_counter: "IdCounter",
    until_days: float,
) -> simpy.events.Event:
    """
    SimPy process: generates arrivals for a single category until time >= until_days.
    """
    while True:
        iat = sample_exponential(rng, mean_iat_days)
        yield env.timeout(iat)

        if env.now >= until_days:
            return

        pid = id_counter.next_id()
        patients_out.append(Patient(id=pid, category=category, arrival_time=float(env.now)))


class IdCounter:
    """Simple incrementing ID generator."""
    def __init__(self, start: int = 1) -> None:
        self._next = start

    def next_id(self) -> int:
        val = self._next
        self._next += 1
        return val


def generate_patients(
    *,
    sim_days: float,
    categories: Optional[Sequence[str]] = None,
    random_seed: Optional[int] = 42,
) -> list[Patient]:
    """
    Run an arrivals-only simulation and return the list of generated patients.

    Params
    ------
    sim_days:
        Length of simulation (days).
    categories:
        Which categories to include. Default: all.
        Valid values: "stroke", "tia", "complex_neuro", "other"
    random_seed:
        Seed for reproducibility.

    Returns
    -------
    List[Patient] with (id, category, arrival_time).
    """
    if sim_days <= 0:
        raise ValueError("sim_days must be > 0")

    selected = list(categories) if categories is not None else list(CATEGORY_ORDER)

    unknown = sorted(set(selected) - set(CATEGORY_ORDER))
    if unknown:
        raise ValueError(f"Unknown categories: {unknown}. Valid: {list(CATEGORY_ORDER)}")

    env = simpy.Environment()
    rng_master = np.random.default_rng(random_seed)

    # One RNG per stream to avoid accidental coupling between processes.
    stream_rngs: dict[str, np.random.Generator] = {
        cat: np.random.default_rng(rng_master.integers(0, 2**32 - 1, dtype=np.uint32).item())
        for cat in selected
    }

    patients: list[Patient] = []
    id_counter = IdCounter(start=1)

    # Start one arrival process per category
    for cat in selected:
        env.process(
            _arrival_stream(
                env,
                category=cat,
                mean_iat_days=MEAN_IAT_DAYS[cat],
                rng=stream_rngs[cat],
                patients_out=patients,
                id_counter=id_counter,
                until_days=sim_days,
            )
        )

    # Run
    env.run(until=sim_days)

    # Sort by arrival time for convenience (optional)
    patients.sort(key=lambda p: p.arrival_time)
    return patients


# ---- Optional quick check / demo ----

def summarise_arrivals(patients: Iterable[Patient]) -> dict[str, int]:
    counts: dict[str, int] = {c: 0 for c in CATEGORY_ORDER}
    for p in patients:
        counts[p.category] += 1
    return counts


if __name__ == "__main__":
    sim_days = 365.0
    patients = generate_patients(sim_days=sim_days)  # default: all categories
    counts = summarise_arrivals(patients)

    total = sum(counts.values()) or 1
    print(f"Simulated {total} arrivals over {sim_days} days")
    for cat in CATEGORY_ORDER:
        print(f"  {cat:14s}: {counts[cat]:5d}  ({counts[cat]/total:6.2%})")

    # Base-period proportions (for comparison)
    base_total = sum(BASE_PERIOD_COUNTS.values())
    print("\nBase-period proportions:")
    for cat in CATEGORY_ORDER:
        print(f"  {cat:14s}: {BASE_PERIOD_COUNTS[cat]/base_total:6.2%}")

Simulated 597 arrivals over 365.0 days
  stroke        :   322  (53.94%)
  tia           :    37  ( 6.20%)
  complex_neuro :   119  (19.93%)
  other         :   119  (19.93%)

Base-period proportions:
  stroke        : 54.01%
  tia           :  6.46%
  complex_neuro : 18.66%
  other         : 20.87%


## Observsations

This appears to generate patient arrivals, however there are some issues.
1. The sampling process does not use a random seed, and therefore will not be reproducible between runs.
2. There is no facility to visualise patient arrivals if desired
3. The data passed to the simulation model is not containerised

These issues are addressed in iteration 2